In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

!pip install --upgrade transformers


In [ ]:
!pip install -q transformers
!pip install -q torch
!pip install -q datasets
!pip install -q sklearn.utils
!pip install -q accelerate
!pip install -q numpy
!pip install -q pandas

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 4.2 MB/s eta 0:00:00


In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [ ]:
from transformers import ModernBertModel, ModernBertConfig, ModernBertForSequenceClassification

import torch, gc
gc.collect()
torch.cuda.empty_cache()


In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    EvalPrediction,
    logging
)

import numpy as np
import os
import json
import random
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
    roc_auc_score
)
from scipy.special import softmax

# --- Configuration ---
SEED = 42
# MODEL_NAME = "answerdotai/ModernBERT-base"
MODEL_NAME="markusbayer/CySecBERT"
# Prefix for file paths within the mounted Google Drive
DRIVE_PREFIX = "/content/drive/MyDrive/266-final-project-data"
TRAIN_FILE = os.path.join(DRIVE_PREFIX, "train_dataset.csv")
VAL_FILE = os.path.join(DRIVE_PREFIX, "val_dataset.csv")
TEST_FILE = os.path.join(DRIVE_PREFIX, "test_dataset.csv")


# Set seed for reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Using random seed: {SEED}")
# Directory to save the final trained model
OUTPUT_DIR = os.path.join(DRIVE_PREFIX, "guardrail_model_v3_CySecBERT")
LOGGING_DIR = os.path.join(OUTPUT_DIR, "logs")

# Define the labels
LABEL_MAP = {
    "Benign": 0,
    "Malicious": 1
}

# Create revered map for the model config
ID2LABEL = {v: k for k, v in LABEL_MAP.items()}

# --- 1. Load Datasets ---
def load_and_prep_dataset(file_path, label_map):
    """Loads a CSV and maps its string labels to integers."""
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: Dataset file not found at {file_path}")
        return None

    df = df.rename(columns={"Obfuscated_Prompt": "text", "Final_Label": "label"})
    df = df.dropna(subset=["text", "label"])

    # Map string labels to integers
    df["label"] = df["label"].map(label_map)
    df = df[df['label'].notna()] # Drop any rows that weren't in the map
    df = df.astype({"label": int})

    return Dataset.from_pandas(df)

# --- 2. Define Metrics Function ---
def compute_metrics(p: EvalPrediction):
    """
    Compute Accuracy, F1, Precision, Recall, ROC-AUC, and AUPRC.
    """
    labels = p.label_ids
    logits = p.predictions

    # Get hard predictions (for Acc, F1, P, R)
    preds = np.argmax(logits, axis=1)

    # Get probabilities for the "Malicious" class (class 1)
    # We apply softmax to the logits to get probabilities
    probs = softmax(logits, axis=1)
    malicious_probs = probs[:, 1]

    # Standard metrics
    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='binary')
    precision = precision_score(labels, preds, average='binary')
    recall = recall_score(labels, preds, average='binary')

    try:
        auprc = average_precision_score(labels, malicious_probs)
    except ValueError: # Happens if only one class is present in a batch
        auprc = 0.0

    try:
        roc_auc = roc_auc_score(labels, malicious_probs)
    except ValueError:
        roc_auc = 0.0

    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "auprc": auprc,
        "roc_auc": roc_auc
    }


def main():
    # --- 1. Setup ---
    # Suppress warnings about a pre-trained model being re-trained
    logging.set_verbosity_error()

    label_map = {"Benign": 0, "Malicious": 1}
    id2label = {0: "Benign", 1: "Malicious"}
    label2id = {"Benign": 0, "Malicious": 1}

    # --- 2. Load Datasets ---
    print("Loading and preparing datasets...")
    train_dataset = load_and_prep_dataset(TRAIN_FILE, label_map)
    val_dataset = load_and_prep_dataset(VAL_FILE, label_map)
    test_dataset = load_and_prep_dataset(TEST_FILE, label_map)

    if not train_dataset or not val_dataset or not test_dataset:
        print("Aborting due to missing dataset files.")
        return

    # --- 3. Load Tokenizer and Model ---
    print(f"Loading tokenizer and model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,
                                              do_lower_case=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=id2label,
        label2id=label2id
    )

    # --- 4. Tokenize Datasets ---
    print("Tokenizing datasets - Use 512 max tokens for consistent model eval")
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

    tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
    tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True)
    tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

    # --- 5. Configure Training ---
    print("Configuring trainer...")

    # Set up early stopping
    early_stopping_callback = EarlyStoppingCallback(
        early_stopping_patience=3,    # Stop if metric doesn't improve for 3 evals
        early_stopping_threshold=0.005 # How much it needs to improve
    )

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        logging_dir=LOGGING_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        warmup_steps=100,
        weight_decay=0.01,

        # Evaluation and Logging
        eval_strategy="steps",       # Use 'eval_strategy' (correct)
        eval_steps=50,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="f1",  # Use F1 as the primary metric
        greater_is_better=True,

        # Housekeeping
        overwrite_output_dir=True,   # Good for re-running in Colab
        save_strategy="steps",       # Match save strategy to eval strategy
        save_steps=50,
        save_total_limit=2,          # Save only the best and the latest
        report_to="tensorboard",
        seed=SEED,
        data_seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[early_stopping_callback] # Add the callback here
    )

    # --- 6. Train ---
    print("\n--- Starting Model Training ---")
    trainer.train()
    print("--- Training Complete ---")

    # --- 7. Evaluate on Test Set ---
    print("\n--- Evaluating on Test Set ---")
    test_results = trainer.evaluate(eval_dataset=tokenized_test_dataset)

    print("\n--- Final Test Results ---")
    for key, value in test_results.items():
        # Print all eval metrics, which now include auprc and roc_auc
        print(f"{key.replace('eval_', '').capitalize():<10}: {value:.4f}")

    # --- 8. Save Artifacts ---
    print(f"\nSaving model, tokenizer, and results to {OUTPUT_DIR}")
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    # Save test results
    test_results_file = os.path.join(OUTPUT_DIR, "test_results.json")
    with open(test_results_file, 'w') as f:
        json.dump(test_results, f, indent=4)

    # Save training history
    history_file = os.path.join(OUTPUT_DIR, "training_log_history.json")
    with open(history_file, 'w') as f:
        # Filter log history to only include eval entries and final test results
        eval_history = [log for log in trainer.state.log_history if 'eval_loss' in log]
        final_log = {"final_test_set_metrics": test_results}
        json.dump({"evaluation_history": eval_history, **final_log}, f, indent=4)

    print("\n--- All Artifacts Saved ---")


if __name__ == "__main__":
    main()

Using random seed: 42
Loading and preparing datasets...
Loading tokenizer and model: markusbayer/CySecBERT
Tokenizing datasets...


Map:   0%|          | 0/27000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Configuring trainer...

--- Starting Model Training ---
{'loss': 0.6635, 'grad_norm': 3.923583507537842, 'learning_rate': 2.45e-05, 'epoch': 0.02962085308056872}
{'eval_loss': 0.5247778296470642, 'eval_accuracy': 0.7553333333333333, 'eval_f1': 0.7422149379536408, 'eval_precision': 0.6979304271246147, 'eval_recall': 0.7925, 'eval_auprc': 0.8269166821790125, 'eval_roc_auc': 0.8619095999999999, 'eval_runtime': 140.0084, 'eval_samples_per_second': 32.141, 'eval_steps_per_second': 0.507, 'epoch': 0.02962085308056872}
{'loss': 0.3699, 'grad_norm': 23.17011070251465, 'learning_rate': 4.9500000000000004e-05, 'epoch': 0.05924170616113744}
{'eval_loss': 0.3393458425998688, 'eval_accuracy': 0.8306666666666667, 'eval_f1': 0.7700663850331925, 'eval_precision': 0.9710806697108066, 'eval_recall': 0.638, 'eval_auprc': 0.9579770665287091, 'eval_roc_auc': 0.9671618000000001, 'eval_runtime': 139.6209, 'eval_samples_per_second': 32.23, 'eval_steps_per_second': 0.509, 'epoch': 0.05924170616113744}
{'loss':